# CNN for Casing Defect Detection (USIT Data)
## Neural networks for energy applications — Day 4, Exercise 4

In this exercise we use a **Convolutional Neural Network** to classify image patches from
Ultrasonic Imager Tool (USIT) casing data. At each depth the USIT measures the casing
wall around 36 or 72 azimuthal sectors, producing a 2-D image (depth × azimuth).

**Learning objectives:**
1. Load and explore well data stored in HDF5 format, and check what the labels are.
2. Visualize USIT thickness maps and the defect flags stored with them.
3. Extract fixed-size image patches and label them as defective / normal.
4. **Split by well**, so that the test measures performance on wells the model has never seen.
5. Build and train a CNN binary classifier in PyTorch, choosing the epoch and the decision
   threshold on validation wells.
6. Evaluate once on test wells with precision, recall, F1 and a confusion matrix, per well,
   and compare with what a random patch split would have reported.

**Read this first: the labels are not ground truth.** The label arrays in the file are called
`all_defects_prediction_V1` and `all_defects_prediction_V2`: they are the outputs of two versions
of an earlier automated defect-detection method. A model trained on them learns to imitate V1,
and every score in this notebook measures **agreement with V1**, not the detection of real
casing defects (Section 1.1).

**Runtime.** About 5–8 minutes on a laptop CPU: two training runs of 12 epochs each.

---
## 0 — Imports and Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

# The patches are small, so a CPU is fast enough. We do not use Apple's "mps" backend here:
# the model's AdaptiveAvgPool2d maps an 8 x 9 feature map to 4 x 4, which MPS does not support.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Setup complete.")

---
## 1 — Load the HDF5 Dataset

The file `data_AutoDefect_USIT_UKPublic.h5` contains **43 groups**. Each group is one
**logging run**, not one well: the group name starts with the well name (for example
`30_17b-A26_dwl_...`), and several wells were logged more than once, including repeat passes
over the same depth interval. The 43 runs come from **18 wells**. Each group stores:

| Key | Shape | Description |
|-----|-------|-------------|
| `THBK` | (n_depth, n_azimuth) | Casing thickness image. Values are centred near zero (a thickness *deviation*, not absolute thickness); the file stores no units. Missing samples are coded **-999.25** (the LAS null value), not NaN |
| `TDEP` | (n_depth,) | Depth, decreasing along the array; the step (15, 30 or 60 file units) matches the `_15B`/`_30B`/`_60B` suffix of the run name |
| `CCLU` | (n_depth,) | Casing collar locator signal |
| `collars` | (n_depth,) | Collar flags (0/1) |
| `all_defects_prediction_V1` | (n_depth, n_azimuth) | Defect flags from version 1 of an earlier automated method (0 = not flagged, 1 or 2 = flagged) |
| `all_defects_prediction_V2` | (n_depth, n_azimuth) | Defect flags from version 2 of that method |

The file has no attributes: no units, no descriptions and no definition of the label values.

In [ ]:
DATA_PATH = "data_AutoDefect_USIT_UKPublic.h5"
NULL_VALUE = -999.25   # code for a missing THBK sample


def well_id(run_name):
    """Well name from a group name, e.g. '30_17b-A26_dwl_DWL_OTHER_46806886_15B' -> '30_17b-A26'."""
    for separator in ("__DWL", "_dwl"):
        if separator in run_name:
            return run_name.split(separator)[0]
    return run_name.rsplit("_", 1)[0]   # the two names without the pattern: drop the sampling suffix


with h5py.File(DATA_PATH, "r") as f:
    run_names = list(f.keys())
    n_attributes = len(f.attrs) + sum(
        len(f[run].attrs) + sum(len(f[run][key].attrs) for key in f[run]) for run in run_names
    )

well_of = {run: well_id(run) for run in run_names}
runs_per_well = pd.Series(well_of).value_counts()

print(f"Groups (logging runs): {len(run_names)}")
print(f"Wells                : {len(runs_per_well)}")
print(f"Attributes in the file (units, descriptions, label definitions): {n_attributes}")
print("\nRuns per well:")
print(runs_per_well.to_string())

In [ ]:
# Summarize each run: size, depth step, missing samples, thickness range and flagged fraction
run_info = []

with h5py.File(DATA_PATH, "r") as f:
    for run in run_names:
        g = f[run]
        thbk = g["THBK"][:]
        missing = thbk == NULL_VALUE
        valid = thbk[~missing]
        v1 = g["all_defects_prediction_V1"][:]
        v2 = g["all_defects_prediction_V2"][:]
        run_info.append({
            "well": well_of[run],
            "run": run,
            "n_depth": thbk.shape[0],
            "n_azimuth": thbk.shape[1],
            "depth_step": float(np.median(np.diff(g["TDEP"][:]))),
            "missing_frac": missing.mean(),
            "thbk_p1": np.percentile(valid, 1),
            "thbk_p99": np.percentile(valid, 99),
            "v1_flagged": np.mean(v1 > 0),
            "v2_flagged": np.mean(v2 > 0),
        })

df_runs = pd.DataFrame(run_info)
with pd.option_context("display.width", 250, "display.precision", 4):
    print(df_runs.to_string(index=False))
print(f"\nTotal depth samples: {df_runs['n_depth'].sum():,}")

# One run is on a different scale from all others: its 1st-99th percentile range (about -1.26 to
# 1.19) is more than ten times wider than that of any other run (at most about -0.14 to 0.06).
# This suggests a different unit or processing, which the file does not document. We leave it out.
EXCLUDED_RUNS = ["ConPr_R03_L002Up_USIT_15B"]
print("\nExcluded:")
print(df_runs.loc[df_runs["run"].isin(EXCLUDED_RUNS), ["run", "thbk_p1", "thbk_p99"]].to_string(index=False))

### 1.1 What the labels are, and what they are not

- The label arrays are named `..._prediction_V1` and `..._prediction_V2`. They are the output
  of two versions of an earlier automated defect-detection method, stored as pixel flags with
  values 0, 1 and 2. The file does not say what 1 and 2 mean; below, flagged pixels are on
  average thinner than unflagged ones, and value 1 more so than value 2.
- Nothing in the file says that the flags were checked against an independent reference
  (for example inspection of recovered casing). They are **not ground truth**.
- The two versions disagree substantially (next cell).

**Consequence for every result in this notebook:** we train a CNN to reproduce V1 from the
THBK image, and precision, recall and F1 measure **agreement with V1**. They do not measure
how well real defects are found. A high score can mean the CNN copies V1's mistakes; a low
score can mean V1 used information that THBK does not show. A claim about real defect
detection would need independently verified labels.

In [ ]:
# Label values versus thickness, and agreement between the two label versions (included runs only)
sum_thbk = {0: 0.0, 1: 0.0, 2: 0.0}
n_pixels = {0: 0, 1: 0, 2: 0}
flagged_both, flagged_either = 0, 0

with h5py.File(DATA_PATH, "r") as f:
    for run in run_names:
        if run in EXCLUDED_RUNS:
            continue
        thbk = f[run]["THBK"][:]
        v1 = f[run]["all_defects_prediction_V1"][:]
        v2 = f[run]["all_defects_prediction_V2"][:]
        valid = thbk != NULL_VALUE
        for value in n_pixels:
            selected = valid & (v1 == value)
            sum_thbk[value] += thbk[selected].sum()
            n_pixels[value] += int(selected.sum())
        flagged_both += np.sum((v1 > 0) & (v2 > 0))
        flagged_either += np.sum((v1 > 0) | (v2 > 0))

for value in n_pixels:
    print(f"V1 value {value}: {n_pixels[value]:>11,} pixels, mean THBK {sum_thbk[value] / n_pixels[value]:+.4f}")
print(f"\nPixels flagged by V1 or V2: {flagged_either:,}; flagged by both: {flagged_both / flagged_either:.1%} of them")

---
## 2 — Visualize USIT Images

The THBK array for each run is essentially a 2-D image where:
- **rows** = depth levels
- **columns** = azimuthal sectors around the casing

We overlay the V1 flags to see where the earlier method marked anomalies. Missing samples
(-999.25) are shown as gaps.

In [ ]:
# Pick a run with a reasonable flagged fraction for visualization
viz_run = df_runs.sort_values("v1_flagged", ascending=False).iloc[2]["run"]
print(f"Visualizing run: {viz_run} (well {well_of[viz_run]})")

with h5py.File(DATA_PATH, "r") as f:
    g = f[viz_run]
    thbk = g["THBK"][:]
    defects = g["all_defects_prediction_V1"][:]
thbk = np.where(thbk == NULL_VALUE, np.nan, thbk)   # missing samples become gaps

# Show a 500-row window for clarity
start = len(thbk) // 4
end = start + 500

fig, axes = plt.subplots(1, 3, figsize=(16, 8))

im0 = axes[0].imshow(thbk[start:end], aspect="auto", cmap="viridis")
axes[0].set_title("THBK (thickness deviation)")
axes[0].set_xlabel("Azimuth Sector")
axes[0].set_ylabel("Depth Index")
plt.colorbar(im0, ax=axes[0], label="THBK (file units)")

im1 = axes[1].imshow(defects[start:end], aspect="auto", cmap="Reds", vmin=0, vmax=2)
axes[1].set_title("V1 flags (earlier method)")
axes[1].set_xlabel("Azimuth Sector")
plt.colorbar(im1, ax=axes[1], label="V1 value")

# Overlay: thickness with flagged pixels highlighted
im2 = axes[2].imshow(thbk[start:end], aspect="auto", cmap="viridis")
defect_mask = np.ma.masked_where(defects[start:end] == 0, defects[start:end])
axes[2].imshow(defect_mask, aspect="auto", cmap="Reds", alpha=0.5)
axes[2].set_title("Thickness + V1 Flags Overlay")
axes[2].set_xlabel("Azimuth Sector")
plt.colorbar(im2, ax=axes[2], label="THBK (file units)")

plt.suptitle(f"Run: {viz_run} (rows {start}-{end})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3 — Extract Image Patches

We slide a window of size `PATCH_HEIGHT × n_azimuth` down each run to create
fixed-size image patches. A patch is labelled **defective (1)** if more than 5% of
its pixels are flagged by V1, otherwise **normal (0)**.

- We standardize all patches to 36 azimuthal columns (down-sample 72-sector runs by
  taking every second column).
- Patches containing missing samples (-999.25) are skipped; the excluded run is left out.
- With a stride of 16 rows, neighbouring 32-row patches **overlap by half**, and repeat runs
  image the same casing again. That is why every patch keeps the name of its **well**: the
  split in Section 5 is made by well.
- Depth sampling differs between runs (see `depth_step`), so a 32-row patch covers different
  lengths of pipe in different runs. We keep this simple setup; resampling to a common depth
  step would be a sensible improvement.

In [ ]:
PATCH_HEIGHT = 32   # depth rows per patch
PATCH_WIDTH = 36    # standardized azimuth sectors
STRIDE = 16         # sliding window stride (neighbouring patches overlap by half)
DEFECT_THRESHOLD = 0.05  # fraction of flagged pixels to label patch as defective


def to_36_sectors(image):
    """Standardize the azimuth axis to 36 sectors (72-sector runs: keep every second column)."""
    if image.shape[1] == 72:
        return image[:, ::2]
    if image.shape[1] != PATCH_WIDTH:
        raise ValueError(f"Unexpected number of azimuth sectors: {image.shape[1]}")
    return image


patches, labels, labels_v2, groups = [], [], [], []
n_skipped = 0

with h5py.File(DATA_PATH, "r") as f:
    for run in run_names:
        if run in EXCLUDED_RUNS:
            continue
        g = f[run]
        thbk = to_36_sectors(g["THBK"][:]).astype(np.float32)
        v1 = to_36_sectors(g["all_defects_prediction_V1"][:])
        v2 = to_36_sectors(g["all_defects_prediction_V2"][:])
        thbk[thbk == NULL_VALUE] = np.nan

        for i in range(0, thbk.shape[0] - PATCH_HEIGHT + 1, STRIDE):
            patch = thbk[i : i + PATCH_HEIGHT, :]

            # Skip patches with missing samples
            if np.isnan(patch).any():
                n_skipped += 1
                continue

            patches.append(patch)
            labels.append(int(np.mean(v1[i : i + PATCH_HEIGHT] > 0) > DEFECT_THRESHOLD))
            labels_v2.append(int(np.mean(v2[i : i + PATCH_HEIGHT] > 0) > DEFECT_THRESHOLD))
            groups.append(well_of[run])

patches = np.array(patches, dtype=np.float32)
labels = np.array(labels, dtype=np.int64)
labels_v2 = np.array(labels_v2, dtype=np.int64)
groups = np.array(groups)

print(f"Total patches: {len(patches):,} (skipped {n_skipped:,} with missing samples)")
print(f"Patch shape: {patches[0].shape}")
print(f"Class distribution (V1): Normal={np.sum(labels == 0):,}, Defective={np.sum(labels == 1):,}")
print(f"Defective ratio: {np.mean(labels):.4f}")

per_well = (pd.DataFrame({"well": groups, "defective": labels})
            .groupby("well")["defective"].agg(patches="size", defective="sum", prevalence="mean"))
print(per_well.sort_values("patches", ascending=False).round(3).to_string())

---
## 4 — Visualize Sample Patches

In [ ]:
# Show 4 normal and 4 defective patches
normal_idx = np.where(labels == 0)[0]
defect_idx = np.where(labels == 1)[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, idx in enumerate(rng.choice(normal_idx, 4, replace=False)):
    axes[0, i].imshow(patches[idx], aspect="auto", cmap="viridis")
    axes[0, i].set_title(f"Normal #{idx} ({groups[idx]})", fontsize=10)
    axes[0, i].axis("off")

for i, idx in enumerate(rng.choice(defect_idx, 4, replace=False)):
    axes[1, i].imshow(patches[idx], aspect="auto", cmap="viridis")
    axes[1, i].set_title(f"Defective #{idx} ({groups[idx]})", fontsize=10)
    axes[1, i].axis("off")

plt.suptitle("Sample Patches (top: normal, bottom: defective according to V1)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5 — Split by Well, then Rebalance and Normalize Using Training Wells Only

**Why by well.** Neighbouring patches overlap by half, repeat runs image the same casing,
and patches from one well share the tool, casing and noise. A random patch split puts
near-copies of training patches into the test set and measures memory, not transfer to a new
well. We therefore split the **wells**: `GroupShuffleSplit` with `groups = well` puts about
70 % of the wells in training, 15 % in validation and 15 % in test.

- **Validation wells** choose the training epoch and the decision threshold.
- **Test wells** are used once, in Section 9, for the locked model and threshold.
- **Rebalancing** (at most 2 normal patches per defective patch) is applied to the training
  wells only; validation and test wells keep their natural prevalence.
- **Normalization** statistics (mean, std) are computed on the training patches only and then
  applied unchanged to validation and test patches.

In [ ]:
# Split WELLS, not patches: first hold out ~15 % of the wells for test,
# then ~15 % of all wells (0.15 / 0.85 of the rest) for validation
all_idx = np.arange(len(labels))
outer_split = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
dev_idx, test_idx = next(outer_split.split(all_idx, labels, groups=groups))

inner_split = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=RANDOM_STATE)
train_pos, val_pos = next(inner_split.split(dev_idx, labels[dev_idx], groups=groups[dev_idx]))
train_idx, val_idx = dev_idx[train_pos], dev_idx[val_pos]

train_wells, val_wells, test_wells = (sorted({str(w) for w in groups[ix]}) for ix in (train_idx, val_idx, test_idx))
assert not set(train_wells) & set(val_wells)
assert not set(train_wells) & set(test_wells)
assert not set(val_wells) & set(test_wells)

split_table = pd.DataFrame({
    "wells": [len(w) for w in (train_wells, val_wells, test_wells)],
    "patches": [len(ix) for ix in (train_idx, val_idx, test_idx)],
    "defective": [int(labels[ix].sum()) for ix in (train_idx, val_idx, test_idx)],
    "prevalence": [labels[ix].mean() for ix in (train_idx, val_idx, test_idx)],
}, index=["train", "validation", "test"])
print(split_table.round(3).to_string())
print("\nValidation wells:", val_wells)
print("Test wells      :", test_wells)

In [ ]:
def rebalance(idx, ratio=2):
    """Keep every defective patch and at most `ratio` normal patches per defective patch."""
    normal = idx[labels[idx] == 0]
    defective = idx[labels[idx] == 1]
    keep = rng.choice(normal, min(ratio * len(defective), len(normal)), replace=False)
    return rng.permutation(np.concatenate([keep, defective]))


def normalise(idx, mean, std):
    """Scale patches with the given (training) statistics and add the channel axis: (N, H, W) -> (N, 1, H, W)."""
    return ((patches[idx] - mean) / (std + 1e-8))[:, np.newaxis, :, :].astype(np.float32)


# 1) Rebalance the TRAINING wells only
train_bal_idx = rebalance(train_idx)
print(f"Training patches after rebalancing: {len(train_bal_idx):,} "
      f"(normal {np.sum(labels[train_bal_idx] == 0):,}, defective {np.sum(labels[train_bal_idx] == 1):,})")

# 2) Normalization statistics from the training patches only
X_mean = patches[train_bal_idx].mean()
X_std = patches[train_bal_idx].std()
print(f"Training mean {X_mean:.6f}, std {X_std:.6f}")

X_train, y_train = normalise(train_bal_idx, X_mean, X_std), labels[train_bal_idx]
X_val, y_val = normalise(val_idx, X_mean, X_std), labels[val_idx]
X_test, y_test = normalise(test_idx, X_mean, X_std), labels[test_idx]

print(f"Train: {len(X_train):,}  Validation: {len(X_val):,}  Test: {len(X_test):,}")
print(f"Input shape: {X_train.shape}")

In [ ]:
# Convert to PyTorch DataLoaders
BATCH_SIZE = 64

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)   # used once, in Section 9

print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}")

---
## 6 — Build the CNN Model

A 3-block CNN architecture:

| Block | Layers |
|-------|--------|
| Conv Block 1 | Conv2d(1, 32, 3, pad=1) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 2 | Conv2d(32, 64, 3, pad=1) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 3 | Conv2d(64, 128, 3, pad=1) → BatchNorm → ReLU → AdaptiveAvgPool(4,4) |
| Classifier | Flatten → Linear(128×4×4, 64) → ReLU → Dropout(0.5) → Linear(64, 2) |

In [ ]:
class DefectCNN(nn.Module):
    """CNN for binary defect detection on USIT image patches."""

    def __init__(self):
        super(DefectCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Conv Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.AdaptiveAvgPool2d((4, 4))

        # Classifier
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


model = DefectCNN().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7 — Train the Model

After every epoch we score the **validation wells** with the **average precision** (AP, the
area under the precision-recall curve). AP does not need a threshold and suits a minority
class; a model with no skill scores the prevalence. We keep a copy of the weights from the
epoch with the best validation AP. The test wells are not used here.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    return running_loss / total, correct / total


def predict_proba(model, loader, criterion, device):
    """Return the predicted defect probability of every patch, the targets and the mean loss."""
    model.eval()
    probs, targets_all, running_loss = [], [], 0.0
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            running_loss += criterion(outputs, targets).item() * images.size(0)
            probs.append(F.softmax(outputs, dim=1)[:, 1].cpu().numpy())
            targets_all.append(targets.cpu().numpy())
    return np.concatenate(probs), np.concatenate(targets_all), running_loss / len(loader.dataset)


print("Training and prediction functions defined.")

In [ ]:
NUM_EPOCHS = 12   # classroom budget; the best epoch is chosen on the validation wells

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_ap": []}
best_ap, best_epoch, best_state = -1.0, 0, None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_prob, val_true, val_loss = predict_proba(model, val_loader, criterion, device)
    val_ap = average_precision_score(val_true, val_prob)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_ap"].append(val_ap)
    if val_ap > best_ap:
        best_ap, best_epoch, best_state = val_ap, epoch, copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val AP: {val_ap:.4f}")

model.load_state_dict(best_state)
print(f"\nKeeping epoch {best_epoch}: validation AP {best_ap:.4f} "
      f"(no-skill level = validation prevalence {y_val.mean():.4f})")

---
## 8 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], "o-", label="Train", color="steelblue")
ax1.plot(epochs, history["val_loss"], "s--", label="Validation wells", color="tomato")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()

ax2.plot(epochs, history["val_ap"], "s--", label="Validation wells", color="tomato")
ax2.axhline(y_val.mean(), color="gray", linestyle=":", label="No skill (prevalence)")
ax2.axvline(best_epoch, color="black", linestyle=":", label=f"Kept epoch ({best_epoch})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Average precision")
ax2.set_ylim(0, 1)
ax2.set_title("Validation Average Precision")
ax2.legend()

plt.suptitle("DefectCNN Training on USIT Patches", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9 — Choose the Threshold on Validation Wells, then Evaluate Once on Test Wells

Training on rebalanced data shifts the predicted probabilities, so 0.5 is not a meaningful
default threshold. We take the threshold with the highest F1 score on the **validation
wells**. (In practice the threshold should follow from the cost of a missed defect versus a
false alarm.)

Then model and threshold are locked, and the **test wells** are evaluated once, at their
natural prevalence. We report the overall scores, two references for reading them (a simple
baseline that scores each patch by its thinnest pixel, and how well the **V2** labels agree
with V1 on the same patches), and the scores per test well. Remember that all these numbers
measure agreement with V1 (Section 1.1).

Reading the per-well table:
- AP measures how well the scores **rank** the patches within a well; precision and recall also
  depend on the **threshold** chosen on the validation wells. A reasonable AP with a very low
  recall means that the scores in that well sit lower than in the validation wells, so the
  threshold does not transfer. For the same reason, the pooled test AP can be lower than the AP
  of every single test well.
- If both the CNN and the thinnest-pixel baseline score close to a well's prevalence, the V1 flags
  there do not correspond to thin pixels that THBK shows (V1 may have used other inputs, or
  flagged something else). A model that sees only THBK cannot reproduce such labels.

In [ ]:
# 9.1 Threshold from the VALIDATION wells: the one with the highest F1
val_prob, val_true, _ = predict_proba(model, val_loader, criterion, device)
precision, recall, thresholds = precision_recall_curve(val_true, val_prob)
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best = int(np.argmax(f1))
THRESHOLD = float(thresholds[best])
print(f"Threshold chosen on validation wells: {THRESHOLD:.3f} "
      f"(validation precision {precision[best]:.3f}, recall {recall[best]:.3f}, F1 {f1[best]:.3f})")

# 9.2 Final evaluation on the TEST wells: once, with the locked model and threshold
test_prob, test_true, _ = predict_proba(model, test_loader, criterion, device)
test_pred = (test_prob >= THRESHOLD).astype(int)

print("\n=== Test wells: agreement with the V1 labels (natural prevalence) ===")
print(f"Prevalence:        {test_true.mean():.4f}  (the AP of a no-skill model)")
print(f"Average precision: {average_precision_score(test_true, test_prob):.4f}")
print(f"Precision:         {precision_score(test_true, test_pred, zero_division=0):.4f}")
print(f"Recall:            {recall_score(test_true, test_pred):.4f}")
print(f"F1 Score:          {f1_score(test_true, test_pred):.4f}")
print()
print(classification_report(test_true, test_pred, target_names=["Normal", "Defective"], zero_division=0))

# 9.3 References: a simple baseline, and the agreement between the two label versions
thinnest_val = -patches[val_idx].min(axis=(1, 2))    # higher score = thinner pixel somewhere in the patch
thinnest_test = -patches[test_idx].min(axis=(1, 2))
print(f"Baseline 'thinnest pixel' AP: validation {average_precision_score(y_val, thinnest_val):.4f}, "
      f"test {average_precision_score(test_true, thinnest_test):.4f}")
v2_test = labels_v2[test_idx]
print(f"V2 labels scored against V1 on the test wells: precision {precision_score(test_true, v2_test):.3f}, "
      f"recall {recall_score(test_true, v2_test):.3f}, F1 {f1_score(test_true, v2_test):.3f}")

# 9.4 Per test well: results usually differ a lot between wells
test_groups = groups[test_idx]
per_well_rows = []
for well in test_wells:
    m = test_groups == well
    has_defects = bool(test_true[m].any())
    per_well_rows.append({
        "well": well,
        "patches": int(m.sum()),
        "prevalence": test_true[m].mean(),
        "AP_cnn": average_precision_score(test_true[m], test_prob[m]) if has_defects else np.nan,
        "AP_thinnest": average_precision_score(test_true[m], thinnest_test[m]) if has_defects else np.nan,
        "precision": precision_score(test_true[m], test_pred[m], zero_division=0),
        "recall": recall_score(test_true[m], test_pred[m], zero_division=0),
        "f1": f1_score(test_true[m], test_pred[m], zero_division=0),
    })
print()
print(pd.DataFrame(per_well_rows).round(3).to_string(index=False))

In [ ]:
# Confusion matrix on the test wells at the validation-chosen threshold
cm = confusion_matrix(test_true, test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Defective"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Test Wells, Threshold {THRESHOLD:.2f} (labels: V1)")
ax.grid(False)
plt.tight_layout()
plt.show()

---
## 10 — Visualize Predictions on Sample Patches

Inspection only: nothing in this section or the next changes the model or the threshold.
(Changing them after looking at test wells would turn the test wells into validation wells.)

In [ ]:
# Show 8 test patches with V1 label vs predicted label
sample_idx = rng.choice(len(X_test), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
class_names = ["Normal", "Defective"]

for i, ax in enumerate(axes.flat):
    idx = sample_idx[i]
    ax.imshow(X_test[idx, 0], aspect="auto", cmap="viridis")
    true_label = class_names[y_test[idx]]
    pred_label = class_names[test_pred[idx]]
    colour = "green" if y_test[idx] == test_pred[idx] else "red"
    ax.set_title(f"V1: {true_label}\nPred: {pred_label} (p={test_prob[idx]:.2f})", fontsize=9, color=colour)
    ax.axis("off")

plt.suptitle("Test Patches: Predictions (green = agrees with V1, red = disagrees)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 11 — Apply the Model to a Held-out Test Well

Slide the model down a run from a **test well** (never used for training, for the epoch or for
the threshold) and build a defect probability log. We take the test well with the most flagged
patches and its longest run, and show the 500-row window with the most V1 flags.

In [ ]:
# A test well: the one with the most flagged patches, and its longest run
test_defective_per_well = pd.Series(test_true, index=test_groups).groupby(level=0).sum()
full_well = test_defective_per_well.idxmax()
full_run = df_runs[df_runs["well"] == full_well].sort_values("n_depth").iloc[-1]["run"]
print(f"Test well: {full_well}, run: {full_run}")

with h5py.File(DATA_PATH, "r") as f:
    thbk_full = to_36_sectors(f[full_run]["THBK"][:]).astype(np.float32)
    defects_full = to_36_sectors(f[full_run]["all_defects_prediction_V1"][:])
thbk_full[thbk_full == NULL_VALUE] = np.nan

# Predict on overlapping windows (same PATCH_HEIGHT and STRIDE as training)
pred_sum = np.zeros(thbk_full.shape[0])
pred_count = np.zeros(thbk_full.shape[0])

model.eval()
with torch.no_grad():
    for i in range(0, thbk_full.shape[0] - PATCH_HEIGHT + 1, STRIDE):
        patch = thbk_full[i : i + PATCH_HEIGHT, :]
        if np.isnan(patch).any():
            continue
        patch_norm = (patch - X_mean) / (X_std + 1e-8)   # TRAINING statistics
        patch_tensor = torch.tensor(patch_norm[np.newaxis, np.newaxis, :, :].astype(np.float32)).to(device)
        defect_prob = F.softmax(model(patch_tensor), dim=1)[0, 1].item()
        pred_sum[i : i + PATCH_HEIGHT] += defect_prob
        pred_count[i : i + PATCH_HEIGHT] += 1

# Average overlapping predictions; rows without a complete window stay empty (NaN)
pred_map = np.full(thbk_full.shape[0], np.nan)
covered = pred_count > 0
pred_map[covered] = pred_sum[covered] / pred_count[covered]

flag_track = np.mean(defects_full > 0, axis=1)

# 500-row window with the most V1 flags
WINDOW = 500
centre = int(np.argmax(np.convolve(flag_track, np.ones(WINDOW), mode="same")))
start = int(np.clip(centre - WINDOW // 2, 0, len(flag_track) - WINDOW))
rows = range(start, start + WINDOW)

fig, axes = plt.subplots(1, 3, figsize=(14, 8), sharey=True)

axes[0].imshow(thbk_full[start:start + WINDOW], aspect="auto", cmap="viridis",
               extent=(0, PATCH_WIDTH, start + WINDOW, start))
axes[0].set_title("THBK Thickness")
axes[0].set_xlabel("Azimuth Sector")
axes[0].set_ylabel("Depth Index")

axes[1].plot(flag_track[start:start + WINDOW], rows, color="red", linewidth=1)
axes[1].set_title("V1 Flag Fraction (earlier method)")
axes[1].set_xlim(0, 1)

axes[2].plot(pred_map[start:start + WINDOW], rows, color="steelblue", linewidth=1)
axes[2].axvline(THRESHOLD, color="black", linestyle=":", label=f"Threshold {THRESHOLD:.2f}")
axes[2].set_title("Predicted Defect Probability")
axes[2].set_xlim(0, 1)
axes[2].legend(loc="lower right")

plt.suptitle(f"Held-out Test Well {full_well}, run {full_run}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 12 — Compare with a Random Patch Split

What would a random patch split have told us? We repeat the recipe (rebalance and normalize
on the training part only, same model, same epoch budget, best epoch by AP) with a **random
80/20 split of the patches of the development wells** (training + validation wells). The test
wells stay untouched.

Compare the held-out AP with the validation-well AP from Section 7 and the test-well AP
from Section 9, each against its own prevalence (the no-skill level).

In [ ]:
# Random 80/20 patch split inside the development wells (the test wells are not used)
rand_train_idx, rand_hold_idx = train_test_split(
    dev_idx, test_size=0.2, random_state=RANDOM_STATE, stratify=labels[dev_idx]
)
rand_bal_idx = rebalance(rand_train_idx)
rand_mean, rand_std = patches[rand_bal_idx].mean(), patches[rand_bal_idx].std()

rand_train_loader = DataLoader(
    TensorDataset(torch.tensor(normalise(rand_bal_idx, rand_mean, rand_std)), torch.tensor(labels[rand_bal_idx])),
    batch_size=BATCH_SIZE, shuffle=True)
rand_hold_loader = DataLoader(
    TensorDataset(torch.tensor(normalise(rand_hold_idx, rand_mean, rand_std)), torch.tensor(labels[rand_hold_idx])),
    batch_size=BATCH_SIZE, shuffle=False)

torch.manual_seed(RANDOM_STATE)
model_random = DefectCNN().to(device)
optimizer_random = optim.Adam(model_random.parameters(), lr=1e-3)
best_random_ap = -1.0
for epoch in range(1, NUM_EPOCHS + 1):
    train_one_epoch(model_random, rand_train_loader, criterion, optimizer_random, device)
    hold_prob, hold_true, _ = predict_proba(model_random, rand_hold_loader, criterion, device)
    best_random_ap = max(best_random_ap, average_precision_score(hold_true, hold_prob))

comparison = pd.DataFrame({
    "evaluated on": ["random held-out patches (development wells)", "validation wells", "test wells (final)"],
    "average precision": [best_random_ap, best_ap, average_precision_score(test_true, test_prob)],
    "prevalence (no skill)": [labels[rand_hold_idx].mean(), y_val.mean(), test_true.mean()],
}, index=["random patch split", "split by well", "split by well"])
print(comparison.round(3).to_string())

---
## Summary

In this exercise we:

1. **Loaded** ultrasonic casing inspection data from an HDF5 file with 43 logging runs from
   18 wells, and found that its labels are the output of an earlier automated method (V1, V2),
   not verified ground truth.
2. **Visualized** USIT thickness maps and the V1 flags, handling the -999.25 missing values.
3. **Extracted** overlapping image patches, labelled them from V1, and kept each patch's well.
4. **Split by well** into training, validation and test wells, rebalanced and normalized using
   the training wells only.
5. **Built** a 3-block CNN with BatchNorm and Dropout for binary classification.
6. **Trained** for 12 epochs, keeping the epoch with the best validation AP, and chose the
   decision threshold on the validation wells.
7. **Evaluated once** on the test wells, overall and per well, next to a simple baseline and
   the agreement between V2 and V1.
8. **Applied** the model to a held-out test well to produce a defect probability log.
9. **Compared** with a random patch split of the development wells.

**Key takeaways:**
- USIT thickness data is naturally structured as 2-D images, making CNNs a natural fit.
- Overlapping patches and repeat runs make random patch splits optimistic: split by well (or
  by the unit you will deploy on) and keep the test wells for one final check.
- Rebalancing, normalization statistics, epochs and thresholds are fitted or chosen on
  training and validation data only; test wells keep their natural prevalence.
- Results vary strongly between wells: report them per well.
- Labels produced by an earlier method limit every claim: the scores here measure agreement
  with V1, not the detection of real defects.